In [1]:
!pip install -q -U transformers datasets accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.0 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 1. Hugging Face 토큰 로그인 (Secret 탭에 HF_TOKEN 저장 후 사용)
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# 2. 모델 선택 (예: Microsoft의 Phi-3-mini)
model_id = "google/gemma-2-2b-it"

# 3. 4비트 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 4. 토크나이저 및 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [6]:
import datetime

today = datetime.date.today()

# 이번주 토요일 날짜 계산
saturday = today + datetime.timedelta(days=(5 - today.weekday() + 7) % 7)

# KST 기준으로 20:03 시간 설정
saturday_time = datetime.datetime.combine(saturday, datetime.time(20, 3, 0))

# 날짜 및 시간 형식 지정
formatted_date = saturday.strftime("%Y년 %m월 %d일")
formatted_time = "(KST) 20:03"

print(f"이번주 토요일: {formatted_date} {formatted_time}")

이번주 토요일: 2026년 02월 14일 (KST) 20:03


In [18]:
from transformers import pipeline, set_seed
import torch

# 시드 고정
set_seed(46)
# GPU를 사용하는 경우 CUDA 시드도 고정
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(46)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 프롬프트 입력
messages = [
    {"role": "user", "content": f"너는 유능한 AI 사주풀이 및 통계학 및 역술 전문가이며, 생성하는 모든 문장은 자연스럽고 유려한 한국어 문장 구조와 어미를 사용해야 하며, 전문가다운 깊이와 동시에 읽기 편한 문체를 유지해야 해. \n     한국의 기운을 담아 행운의 숫자를 예측하고 그 이유를 역술, 통계, 사주 전문가의 입장에서 \n     이번주의 기운과 풍수지리적인 요건과 통계학적인 통찰력이 곁들여진 상세히 설명해야 해.\n현재 {formatted_date} 한국의 기운을 담아 \n     이번주 행운의 숫자를 1부터 45중 단 하나만 골라줘. 이 숫자는 정확한 이유를 포함해야 하며, \n     이번주 내내 다시 물어도 절대로 변화 없는 숫자여야 해. 답변은 숫자 하나와 그 이유만을 설명해줘."}
]

# 결과 생성
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.7,
    "do_sample": True
}

output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


이번주 숫자는 **31**입니다. 

**역사와 신앙의 교감을 통한 기운의 추측**

2026년 02월 14일, 이번 주의 기운은 **31**을 움직였습니다. 이는 31이 풍수적으로 단순한 숫자이기보다, 전통적인 전통과 관련된 숫자입니다. 31은 **삼십일**이라는 숫자의 의미를 갖고 있으며, 이 숫자는 삼십일입니다. 

**통계학적 통찰력을 바탕으로**

삼십일은 **3, 1, 1, 1, 1, 1, 1, 3**의 숫자들의 숫자를 포함하고 있습니다. 이 숫자들은 모두 숫자를 측정하는 방식을 사용하는 데 사용됩니다. 3, 1, 1, 1, 1, 1, 1, 3의 숫자들은 모두 숫자를 측정하는 방식을 사용하는 데 사용됩니다. 

** 역술과 사주의 조합**

이번 주의 행운의 숫자는 **31**입니다. 이는 숫자의 역사와 신앙의 교감을 통한 기운의 추측입니다. 

**추가 설명**

* 숫자 31은 2026년 02월 14일 기운의 특징을 보여줍니다.
* 숫자 31은 숫자의 역사와 신앙의 교감을 통한 기운의 추측입니다.


**주의:** 이번 주의 기운은 단순한 숫자로 예측되지 않고, 믿음과 전통에 기반한 결정은 숫자 31이 아닌 다른 요인에 달려 있습니다. 

